# Fuel demand from vessel telemetry

This notebook models fuel demand from a vessel's telemetry stream, two ways. The first is
inference: an OLS fit with HC3 standard errors and an elastic net fit on the same design, to see
which signals carry the effect. The second is prediction: six sklearn pipelines compared by
cross-validation, the best one tuned with random search, then scored on the final 20 percent of
the voyage. Residuals are mapped along the track so a systematic error over one leg is visible.

The telemetry is synthetic: 6,000 one-minute records along a transatlantic route with speed,
heading, draft, wind and wave state, sea temperature, engine load and a fuel demand column that
is a nonlinear function of those with noise. Column names are read from the table, so a real
export with a different set of sensors needs only a different `target`.

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
warnings.filterwarnings("ignore")

DATA = ROOT / "data" / "maritime"
EXPORTS = ROOT / "exports" / "maritime"
DATA.mkdir(parents=True, exist_ok=True)
EXPORTS.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("repo root:", ROOT)

In [ ]:
from datasets.synthetic import synthetic_vessel_telemetry
from fortress_gis.domains import maritime as mar
from fortress_gis.viz.kepler import KeplerMapBuilder, kepler_available

source = DATA / "vessel_telemetry.parquet"
if not source.exists():
    synthetic_vessel_telemetry(6000).to_parquet(source)
telemetry = pd.read_parquet(source)
print(telemetry.shape)
telemetry.head()

## Build the design

`prepare_design` converts the timestamp, separates numeric from categorical columns (a column
with 20 or fewer distinct values is treated as categorical), one-hot encodes the categoricals
with the first level dropped, and returns the encoded frame with the feature list. Coordinates
and the timestamp are excluded from the predictors by default.

In [ ]:
TARGET = "fuel_demand_kg_h"
design = mar.prepare_design(telemetry, target=TARGET)
print("numeric:", design.numeric)
print("categorical:", design.categorical)
print("features after encoding:", len(design.features))
design.frame[design.features].describe().T.round(2).head(12)

## Run the pipeline

`run_maritime_pipeline` fits everything in one call and returns a `MaritimeResult`: the design,
the OLS and elastic net fits, the model leaderboard, the tuned best model with its search
results, holdout metrics, permutation importances, and holdout residuals joined to the
coordinates. The sections below unpack it. Tuning runs `n_iter * cv` fits in parallel across
cores; with 12 candidates and 5 folds this cell takes about a minute on 12 cores.

In [ ]:
result = mar.run_maritime_pipeline(telemetry, target=TARGET, cv=5, n_iter=12)
result.summary().round(3)

## Inference: OLS and elastic net

The OLS coefficients answer "how much does fuel demand move per unit of each predictor, holding
the rest fixed". The elastic net (alpha 0.05, L1 weight 0.05) is fit on z-scored features so the
penalty treats them equally, and it shrinks coefficients toward zero. To compare the two, the
OLS estimates are multiplied by each feature's standard deviation, which puts both on the
"kg/h per one standard deviation" scale. Collinear pairs (speed over ground and through water,
wind speed and true wind speed) trade weight between them under the penalty, so read those as
groups rather than one at a time. The penalised fit has no standard errors, so its table has
coefficients only.

In [ ]:
ols, penalised = result.ols, result.penalised
print(ols.metrics())
coef = ols.coefficients().set_index("term")
coef.sort_values("p_value").round(4).head(12)

In [ ]:
pen = penalised.coefficients().set_index("term")
scale = pd.Series(penalised.extra["feature_scale"])
compare = pd.DataFrame(
    {
        "ols_per_sd": coef["estimate"].drop("const") * scale,
        "elastic_net_per_sd": pen["estimate"].drop("const", errors="ignore"),
    }
)
compare.sort_values("ols_per_sd", key=abs, ascending=False).round(3)

## Prediction: compare, tune, hold out

The pipeline cross-validates linear, ridge, lasso, elastic net, random forest and gradient
boosting pipelines (each with imputation and scaling inside the pipeline), takes the best by
RMSE, runs a random search over its hyperparameters, and scores the tuned model on the last 20
percent of rows. The split is by position in the voyage rather than random; adjacent one-minute
records are near copies of each other, and a random split would score the model on its own
training neighbours.

In [ ]:
best = result.best_model
print("best by CV:", best)
result.leaderboard[["model", "rmse_cv", "mae_cv", "r2_cv", "fit_time_s"]].round(3)

In [ ]:
print("holdout metrics")
print(result.holdout_metrics.round(3))
result.tuning_results.sort_values("rank_test_score")[["params", "mean_test_score", "std_test_score"]].head(5)

## Which signals the model uses

Permutation importance shuffles one feature at a time on the holdout and records how much the
score drops. Unlike tree impurity importances it is comparable across model types, and it is
measured on data the model did not train on.

In [ ]:
top = result.importances.head(10)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top["feature"][::-1], top["importance_mean"][::-1], xerr=top["importance_std"][::-1], color="#4c72b0")
ax.set_xlabel("drop in R^2 when shuffled")
ax.set_title(f"Permutation importance, {best}")
plt.show()

## Residuals along the track

Holdout residuals are attached to the coordinates of each record. A residual that drifts with
position over the last leg points at a missing covariate (current, fouling, a sensor offset)
rather than noise.

In [ ]:
res = result.residuals
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(res["observed"].to_numpy(), label="observed", color="#333333")
ax.plot(res["predicted"].to_numpy(), label="predicted", color="#d62728", alpha=0.8)
ax.set_title("Holdout: observed and predicted fuel demand")
ax.set_ylabel("kg/h")
ax.legend()
plt.show()

## Kepler.gl map

The cell below renders the map inline. If the widget shows as blank text, run
`jupyter nbextension enable --py --sys-prefix keplergl` once in the environment and reload the
page; `fortress_gis.viz.kepler.enable_nbextension()` prints the same command. The HTML export in
the last section does not need the extension and opens in any browser.

In [ ]:
track = mar.voyage_track(telemetry)
builder = KeplerMapBuilder(title="Voyage residuals", height=550)
builder.add_layer(track, "track", opacity=0.5)
builder.add_layer(
    res,
    "holdout residuals",
    color_field="residual",
    colors=("#2166ac", "#f7f7f7", "#b2182b"),
    color_scale="quantile",
    radius=5,
)
builder.widget() if kepler_available() else print("keplergl not installed")

## Export

The bundle has the voyage track, the telemetry points with the target, and the holdout residual
points with a diverging QML style. Tables (leaderboard, coefficients, importances) go alongside
as CSV.

In [ ]:
paths_out = mar.export_artifacts(result, telemetry, EXPORTS, name="maritime")
for k, v in paths_out.items():
    print(f"{k:>12}: {v.relative_to(ROOT)}")